🎬 🎓 SCENARIO: “College Smart Assistant System”
🏫 Background Story

A college builds an AI-powered student assistant.

👉 Students can ask:

“What is my attendance?”
“What are my marks?”

👉 Instead of manually checking portals,
👉 AI fetches it instantly.

In [1]:

# ================================
# STEP 1: Install Gradio
# ================================
# !pip install gradio


# ================================
# STEP 2: Dummy Database (Tool Layer)
# ================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ================================
# STEP 3: Tool Functions (MCP Tools)
# ================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


# ================================
# STEP 4: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 5: MCP Agent Logic
# ================================
def mcp_agent(message, student_id, history):

    # Simulate logged-in user (for demo)
    user_id = student_id

    # Security Check
    if not secure_access(user_id, student_id):
        return "🚫 Access Denied", history

    # Tool Invocation Logic
    message_lower = message.lower()

    if "attendance" in message_lower:
        response = get_attendance(student_id)

    elif "marks" in message_lower:
        response = get_marks(student_id)

    elif "hello" in message_lower or "hi" in message_lower:
        response = "👋 Hello! Ask me about attendance or marks."

    else:
        response = "🤖 I can help with attendance or marks."

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history


# ================================
# STEP 6: Gradio Chat UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🎓 Student MCP Agent (Gradio Version)")

    student_id = gr.Textbox(label="Enter Student ID (e.g., 101)")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# STEP 7: Launch App (Public URL)
# ================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# STUDENT MCP USING GROQ INTEGRATION

In [10]:
# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq gradio


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
from dotenv import load_dotenv
import os
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Database (Tool Layer)
# ======================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ======================================
# STEP 4: Tool Functions
# ======================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


# ======================================
# STEP 5: MCP Tool Decision via LLM
# ======================================
def decide_tool(query):
    try:
        prompt = f"""
        You are an AI assistant.

        Decide which function to call:
        - get_attendance
        - get_marks

        Rules:
        - If user asks about attendance → get_attendance
        - If user asks about marks → get_marks

        Only return function name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        tool = response.choices[0].message.content.strip().lower()

        return tool

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 6: MCP Agent (CORE LOGIC)
# ======================================
def mcp_agent(message, student_id, history):

    # Validate input
    if not student_id:
        response = "⚠️ Please enter Student ID"
        history.append((message, response))
        return "", history

    # Step 1: LLM decides tool
    tool = decide_tool(message)

    # Step 2: Tool Invocation
def mcp_agent(message, student_id, history):

    if not student_id:
        response = "⚠️ Please enter Student ID"
        history = history + [
            {"role": "assistant", "content": response}
        ]
        return "", history, history

    tool = decide_tool(message)

    if "attendance" in tool:
        response = get_attendance(student_id)

    elif "marks" in tool:
        response = get_marks(student_id)

    elif tool == "fallback":
        if "attendance" in message.lower():
            response = get_attendance(student_id)
        elif "marks" in message.lower():
            response = get_marks(student_id)
        else:
            response = "⚠️ LLM failed"

    else:
        response = "🤖 I can help with attendance or marks."


    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history, history


# ======================================
# STEP 7: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🚀 MCP Agent with Groq")

    student_id = gr.Textbox(label="Enter Student ID (101 / 102)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot, state]
    )


# ======================================
# STEP 8: Launch App
# ======================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


SCENARIO: “Hospital Smart Assistant System”
🏥 Background Story
A large hospital deploys an AI-powered patient assistant.
👉 Patients can ask:
- “What is my appointment schedule?”
- “What are my latest test results?”
👉 Instead of calling reception or logging into multiple portals,
👉 AI fetches it instantly, providing secure, real-time updates.

In [11]:
# ================================
# STEP 1: Install Gradio
# ================================
# !pip install gradio


# ================================
# STEP 2: Dummy Database (Tool Layer)
# ================================
patients = {
    "P101": {
        "name": "Rahul",
        "appointment": "10 Oct, 10:30 AM - Dr. Sharma",
        "test_results": "Blood Test: Normal"
    },
    "P102": {
        "name": "Priya",
        "appointment": "12 Oct, 2:00 PM - Dr. Mehta",
        "test_results": "X-Ray: Minor fracture"
    }
}


# ================================
# STEP 3: Tool Functions (MCP Tools)
# ================================
def get_appointment(patient_id):
    if patient_id in patients:
        return f"📅 Appointment: {patients[patient_id]['appointment']}"
    return "❌ Patient not found"


def get_test_results(patient_id):
    if patient_id in patients:
        return f"🧪 Test Results: {patients[patient_id]['test_results']}"
    return "❌ Patient not found"


# ================================
# STEP 4: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 5: MCP Agent Logic
# ================================
def hospital_agent(message, patient_id, history):

    # Simulate logged-in user
    user_id = patient_id

    # Security Check
    if not secure_access(user_id, patient_id):
        return "🚫 Access Denied", history

    message_lower = message.lower()

    # Tool Invocation Logic
    if "appointment" in message_lower or "schedule" in message_lower:
        response = get_appointment(patient_id)

    elif "test" in message_lower or "result" in message_lower:
        response = get_test_results(patient_id)

    elif "hello" in message_lower or "hi" in message_lower:
        response = "👋 Hello! Ask me about appointments or test results."

    else:
        response = "🤖 I can help with appointments or test results."

    # Store history (same format as your code)
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history,history


# ================================
# STEP 6: Gradio Chat UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Hospital MCP Agent (Gradio Version)")

    patient_id = gr.Textbox(label="Enter Patient ID (e.g., P101)")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        hospital_agent,
        inputs=[msg, patient_id, state],
        outputs=[msg, chatbot,state]
    )


# ================================
# STEP 7: Launch App
# ================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


SCENARIO: “Banking Smart Assistant System”
🏦 Background Story
A major bank launches an AI-powered customer assistant.
👉 Customers can ask:
- “What is my account balance?”
- “Show me my last 5 transactions.”
- “When is my loan EMI due?”
👉 Instead of logging into apps or waiting on customer service calls,
👉 AI fetches the information instantly, securely, and in real time.

⚙️ Core Idea:
Just like the hospital and college scenarios, the assistant removes manual checking, centralizes financial data, and makes access instant.
💡 Impact:
- Saves customers time.
- Reduces load on call centers.
- Provides personalized financial insights on demand.

In [13]:
# SCENARIO: “AI Banking Assistant with Role-Based Access”
# 🏦 Background Story

# A bank builds an AI assistant to help users:

# Check account balance
# View transactions
# Approve loans
# Manage customer accounts

# 👉 But not everyone can do everything

# 🧑‍🤝‍🧑 Roles in the Bank
# Role	Description
# 👤 Customer	Bank account holder
# 👨‍💼 Employee	Bank staff
# 🧑‍💼 Manager	Branch manager
# 🔐 Permissions (RBAC)
# Action	Customer	Employee	Manager
# View own balance	✅	✅	✅
# View others' accounts	❌	✅	✅
# Approve loan	❌	❌	✅
# View all transactions	❌	✅	✅


# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq gradio


# ======================================
# STEP 2: Load API Key from env file
# ======================================
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("❌ GROQ_API_KEY not found in env files")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Bank Database
# ======================================
accounts = {
    "1001": {"name": "Amit", "balance": 50000},
    "1002": {"name": "Neha", "balance": 75000}
}


# ======================================
# STEP 4: Tool Functions (APIs)
# ======================================
def get_balance(account_id):
    if account_id in accounts:
        return f"💰 Balance of {account_id}: ₹{accounts[account_id]['balance']}"
    return "❌ Account not found"


def approve_loan(account_id):
    if account_id in accounts:
        return f"🏦 Loan approved for account {account_id}"
    return "❌ Account not found"


# ======================================
# STEP 5: RBAC Security Layer
# ======================================
def secure_access(role, user_account, requested_account, action):

    # Manager → full access
    if role == "manager":
        return True

    # Employee → can view all but cannot approve loans
    elif role == "employee":
        if action == "approve_loan":
            return False
        return True

    # Customer → only own account, no loan approval
    elif role == "customer":
        return user_account == requested_account and action != "approve_loan"

    return False


# ======================================
# STEP 6: MCP Tool Decision via LLM
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        You are an AI banking assistant.

        Decide which action to take:
        - get_balance
        - approve_loan

        Rules:
        - Balance related queries → get_balance
        - Loan approval queries → approve_loan

        Return ONLY the action name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        action = response.choices[0].message.content.strip().lower()
        return action

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 7: MCP Agent (Core Logic)
# ======================================
def banking_agent(message, role, user_account, requested_account, history):

    # Input validation
    if not user_account:
        response = "⚠️ Please enter your account ID"
        history.append((message, response))
        return "", history

    if not requested_account:
        requested_account = user_account  # default to own account

    # Step 1: LLM decides action
    action = decide_action(message)

    # Step 2: RBAC Security Check
    if not secure_access(role, user_account, requested_account, action):
        response = "🚫 Access Denied: You are not authorized"

    else:
        # Step 3: Tool Invocation
        if "balance" in action:
            response = get_balance(requested_account)

        elif "loan" in action:
            response = approve_loan(requested_account)

        # Fallback if LLM fails
        elif action == "fallback":
            msg_lower = message.lower()
            if "balance" in msg_lower:
                response = get_balance(requested_account)
            elif "loan" in msg_lower:
                response = approve_loan(requested_account)
            else:
                response = "⚠️ Could not understand request"

        else:
            response = "🤖 Try asking about balance or loan"

    # Save chat history
    history=history+ [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history,history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 AI Banking Assistant (MCP + RBAC + Groq)")

    role = gr.Dropdown(
        ["customer", "employee", "manager"],
        label="Select Role"
    )

    user_account = gr.Textbox(label="Your Account ID (e.g., 1001)")
    requested_account = gr.Textbox(label="Target Account ID (optional)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, role, user_account, requested_account, state],
        outputs=[msg, chatbot,state]
    )


# ======================================
# STEP 9: Launch App
# ======================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


SCENARIO: “University Smart Assistant with Role-Based Access”
🏫 Background Story
A university deploys an AI-powered academic assistant to help students, faculty, and administrators.
👉 Users can ask:
- “What is my attendance record?”
- “Show me my exam results.”
- “Update course schedules.”
- “Approve new course registrations.”
👉 But not everyone can do everything — access depends on roles.

In [17]:
# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq gradio python-dotenv


# ======================================
# STEP 2: Load API Key
# ======================================
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Database (10 Records)
# ======================================
students = {
    "S101": {"name": "Rahul", "attendance": 85, "marks": 78, "course": "AI"},
    "S102": {"name": "Priya", "attendance": 92, "marks": 88, "course": "DS"},
    "S103": {"name": "Amit", "attendance": 75, "marks": 70, "course": "ML"},
    "S104": {"name": "Sneha", "attendance": 88, "marks": 82, "course": "AI"},
    "S105": {"name": "Karan", "attendance": 65, "marks": 60, "course": "Cyber"},
    "S106": {"name": "Anjali", "attendance": 90, "marks": 91, "course": "DS"},
    "S107": {"name": "Rohit", "attendance": 78, "marks": 74, "course": "ML"},
    "S108": {"name": "Neha", "attendance": 95, "marks": 93, "course": "AI"},
    "S109": {"name": "Vikas", "attendance": 70, "marks": 65, "course": "Cyber"},
    "S110": {"name": "Pooja", "attendance": 89, "marks": 87, "course": "DS"}
}

courses = {
    "AI": {"schedule": "Mon-Wed 10AM"},
    "DS": {"schedule": "Tue-Thu 2PM"},
    "ML": {"schedule": "Mon-Fri 11AM"},
    "Cyber": {"schedule": "Sat 9AM"}
}


# ======================================
# STEP 4: Tool Functions
# ======================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


def update_schedule(course):
    if course in courses:
        courses[course]["schedule"] = "Updated Schedule ✅"
        return f"📅 Schedule updated for {course}"
    return "❌ Course not found"


def approve_registration(student_id):
    if student_id in students:
        return f"✅ Registration approved for {student_id}"
    return "❌ Student not found"


# ======================================
# STEP 5: RBAC Security Layer
# ======================================
def secure_access(role, user_id, requested_id, action):

    # Admin → full access
    if role == "admin":
        return True

    # Faculty → cannot approve registration
    elif role == "faculty":
        if action == "approve_registration":
            return False
        return True

    # Student → only own data, no update/approve
    elif role == "student":
        if action in ["update_schedule", "approve_registration"]:
            return False
        if user_id != requested_id:
            return False
        return True

    return False


# ======================================
# STEP 6: LLM Decision (MCP Brain)
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        Decide action:
        - get_attendance
        - get_marks
        - update_schedule
        - approve_registration

        Rules:
        attendance → get_attendance
        marks/result → get_marks
        schedule update → update_schedule
        approve/register → approve_registration

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip().lower()

    except:
        return "fallback"


# ======================================
# STEP 7: MCP Agent (CORE)
# ======================================
def university_agent(message, role, user_id, requested_id, history):

    if not user_id:
        response = "⚠️ Enter your ID"
        history = history + [
            {"role": "assistant", "content": response}
        ]
        return "", history, history

    if not requested_id:
        requested_id = user_id

    # Step 1: LLM decides action
    action = decide_action(message)

    # Step 2: RBAC Check
    if not secure_access(role, user_id, requested_id, action):
        response = "🚫 Access Denied"

    else:
        # Step 3: Tool Execution
        if "get_attendance" in action:
            response = get_attendance(requested_id)

        elif "get_marks" in action:
            response = get_marks(requested_id)

        elif "update_schedule" in action:
            course = students.get(requested_id, {}).get("course", "")
            response = update_schedule(course)

        elif "approve_registration" in action:
            response = approve_registration(requested_id)

        elif action == "fallback":
            msg = message.lower()
            if "attendance" in msg:
                response = get_attendance(requested_id)
            elif "marks" in msg:
                response = get_marks(requested_id)
            else:
                response = "⚠️ Could not understand request"

        else:
            response = "🤖 Ask about attendance, marks, schedule, or registration"

    # Step 4: Save chat history (MESSAGE FORMAT ✅)
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history, history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🎓 University Smart Assistant (RBAC + MCP + Groq)")

    role = gr.Dropdown(["student", "faculty", "admin"], label="Role")

    user_id = gr.Textbox(label="Your ID (S101 etc.)")
    requested_id = gr.Textbox(label="Target Student ID (optional)")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        university_agent,
        inputs=[msg, role, user_id, requested_id, state],
        outputs=[msg, chatbot, state]
    )


# ======================================
# STEP 9: Launch
# ======================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


In [19]:
# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq gradio python-dotenv


# ======================================
# STEP 2: Load API Key
# ======================================
import os
from dotenv import load_dotenv
load_dotenv()

from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# ======================================
# STEP 3: Dummy Database (10 Records)
# ======================================
customers = {
    "C101": {"name": "Rahul", "purchases": ["Shoes", "T-shirt"]},
    "C102": {"name": "Priya", "purchases": ["Bag", "Watch"]},
    "C103": {"name": "Amit", "purchases": ["Laptop"]},
    "C104": {"name": "Sneha", "purchases": ["Dress"]},
    "C105": {"name": "Karan", "purchases": ["Shoes", "Cap"]},
    "C106": {"name": "Anjali", "purchases": ["Phone"]},
    "C107": {"name": "Rohit", "purchases": ["Headphones"]},
    "C108": {"name": "Neha", "purchases": ["Makeup Kit"]},
    "C109": {"name": "Vikas", "purchases": ["Keyboard"]},
    "C110": {"name": "Pooja", "purchases": ["Tablet"]}
}

inventory = {
    "Shoes": 50,
    "T-shirt": 100,
    "Laptop": 10,
    "Phone": 25,
    "Tablet": 15
}


# ======================================
# STEP 4: Tool Functions
# ======================================
def get_purchase_history(customer_id):
    if customer_id in customers:
        return f"🛒 Purchases: {customers[customer_id]['purchases']}"
    return "❌ Customer not found"


def check_inventory(product):
    return f"📦 Stock for {product}: {inventory.get(product, 0)}"


def approve_supplier_order():
    return "✅ Supplier order approved"


def manage_schedule():
    return "📅 Employee schedule updated"


# ======================================
# STEP 5: RBAC (FIXED)
# ======================================
def secure_access(role, user_id, requested_id, action):

    action = action.lower().strip()

    # 🧑 Customer
    if role == "customer":
        if "purchase" in action:
            return user_id == requested_id
        return False

    # 👨‍💼 Staff
    elif role == "staff":
        if "purchase" in action or "inventory" in action:
            return True
        return False

    # 🧑‍💼 Manager
    elif role == "manager":
        return True

    return False


# ======================================
# STEP 6: LLM Decision
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        Decide action:
        - get_purchase_history
        - check_inventory
        - approve_supplier_order
        - manage_schedule

        Rules:
        purchase → get_purchase_history
        inventory/stock → check_inventory
        approve → approve_supplier_order
        schedule → manage_schedule

        Query: {query}
        """

        res = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        return res.choices[0].message.content.strip().lower()

    except:
        return "fallback"


# ======================================
# STEP 7: MCP Agent (CORE)
# ======================================
def retail_agent(message, role, user_id, requested_id, history):

    if not user_id:
        response = "⚠️ Enter your Customer ID"
        history = history + [
            {"role": "assistant", "content": response}
        ]
        return "", history, history

    if not requested_id:
        requested_id = user_id

    action = decide_action(message)

    # 🔐 RBAC check
    if not secure_access(role, user_id, requested_id, action):
        response = "🚫 Access Denied"

    else:
        # 🧠 Tool execution (robust matching)
        if "purchase" in action:
            response = get_purchase_history(requested_id)

        elif "inventory" in action:
            words = message.split()
            product = words[-1]
            response = check_inventory(product)

        elif "approve" in action:
            response = approve_supplier_order()

        elif "schedule" in action:
            response = manage_schedule()

        elif action == "fallback":
            msg = message.lower()
            if "purchase" in msg:
                response = get_purchase_history(requested_id)
            elif "inventory" in msg:
                response = check_inventory("Shoes")
            else:
                response = "⚠️ Could not understand"

        else:
            response = "🤖 Ask about purchases, inventory, orders, or schedules"

    # 💬 Save chat history (message format)
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history, history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏬 Retail Smart Assistant (RBAC + MCP)")

    role = gr.Dropdown(["customer", "staff", "manager"], label="Role")

    user_id = gr.Textbox(label="Your ID (C101 etc.)")
    requested_id = gr.Textbox(label="Target Customer ID (optional)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        retail_agent,
        inputs=[msg, role, user_id, requested_id, state],
        outputs=[msg, chatbot, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


SCENARIO: “Corporate Research Assistant System”
🏢 Background Story
A multinational company deploys an AI-powered business intelligence assistant.
👉 Employees can ask:
- “What’s the latest news about Tesla?”
- “What’s Tesla’s current stock price?”
- “Give me a company profile instantly.”
👉 Instead of manually searching news sites, finance portals, and HR databases,
👉 AI fetches all the data in parallel, analyzes it, and generates a professional report.

⚙️ How it works (mapped to your pipeline):
- Parallel Data Collection → AI gathers news, stock prices, and company profiles simultaneously.
- LLM Analysis → AI interprets the combined data, highlighting key insights.
- Report Generation → AI produces a polished, executive-ready report.

💡 Impact:
- Saves analysts hours of manual research.
- Provides real-time, consolidated insights for decision-making.
- Empowers managers with instant reports for board meetings or investor updates.

In [21]:
# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq gradio nest_asyncio python-dotenv


# ======================================
# STEP 2: Setup API
# ======================================
import os
from dotenv import load_dotenv
load_dotenv()

from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# ======================================
# STEP 3: MOCK TOOLS
# ======================================
import asyncio
import random
import nest_asyncio

nest_asyncio.apply()

async def web_search(query):
    await asyncio.sleep(1)
    return f"📰 News about {query}: Market is growing fast."

async def get_stock_data(company):
    await asyncio.sleep(1)
    price = random.randint(100, 500)
    return f"📈 Stock price of {company}: ${price}"

async def fetch_company_profile(company):
    await asyncio.sleep(1)
    return f"👥 {company} has 5000 employees, HQ in USA"


# ======================================
# STEP 4: PARALLEL EXECUTION
# ======================================
async def parallel_research(company):

    results = await asyncio.gather(
        web_search(company),
        get_stock_data(company),
        fetch_company_profile(company),
        return_exceptions=True
    )

    news, stock, profile = results

    return f"""
    News: {news}
    Stock: {stock}
    Profile: {profile}
    """


# ======================================
# STEP 5: LLM CHAIN
# ======================================
def analyse_text(text):
    res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"Analyze and give key insights:\n{text}"
        }]
    )
    return res.choices[0].message.content


def generate_report(analysis, company):
    res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"Create a professional report for {company}:\n{analysis}"
        }]
    )
    return res.choices[0].message.content


# ======================================
# STEP 6: FULL PIPELINE
# ======================================
async def full_pipeline(company):

    data = await parallel_research(company)

    analysis = analyse_text(data)
    report = generate_report(analysis, company)

    return report


# ======================================
# STEP 7: MCP CHAT AGENT
# ======================================
def research_agent(message, history):

    company = message.strip()

    if not company:
        response = "⚠️ Enter a company name"
        history += [{"role": "assistant", "content": response}]
        return "", history, history

    # Run async pipeline
    report = asyncio.run(full_pipeline(company))

    history += [
        {"role": "user", "content": message},
        {"role": "assistant", "content": report}
    ]

    return "", history, history


# ======================================
# STEP 8: GRADIO UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 📊 AI Research Assistant (MCP + Async + Groq)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Enter Company Name (e.g., Tesla)")

    state = gr.State([])

    msg.submit(
        research_agent,
        inputs=[msg, state],
        outputs=[msg, chatbot, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


🏥 SCENARIO: “Healthcare Research Assistant System”
🩺 Background Story
A medical research institute deploys an AI-powered clinical intelligence assistant.
👉 Researchers and doctors can ask:
• 	“What’s the latest research on diabetes treatments?”
• 	“Summarize recent clinical trial results for cancer drugs.”
• 	“Give me a profile of a pharmaceutical company instantly.”
👉 Instead of manually searching journals, trial databases, and company reports,
👉 AI fetches all the data in parallel, analyzes it, and generates a professional research summary.

⚙️ How it works (mapped to your pipeline):
• 	Parallel Data Collection → AI gathers medical news, trial results, and company profiles simultaneously.
• 	LLM Analysis → AI interprets the combined data, highlighting key medical insights.
• 	Report Generation → AI produces a polished, researcher-ready report.

In [22]:
# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq nest_asyncio


# ======================================
# STEP 2: Load Groq API Key (Colab Secret)
# ======================================
from dotenv import load_dotenv
import os
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: MOCK TOOLS (Healthcare MCP Tools)
# ======================================
import asyncio
import random
import nest_asyncio

nest_asyncio.apply()

async def fetch_medical_news(topic):
    await asyncio.sleep(1)
    return f"📰 Latest research on {topic}: New therapies show promising results."

async def fetch_clinical_trials(topic):
    await asyncio.sleep(1)
    success_rate = random.randint(60, 95)
    return f"🧪 Clinical trials for {topic}: Success rate around {success_rate}% in recent studies."

async def fetch_pharma_profile(company):
    await asyncio.sleep(1)
    return f"💊 {company} is a leading pharma company with global clinical research operations."


# ======================================
# STEP 4: PARALLEL TOOL INVOCATION
# ======================================
async def parallel_research(topic):

    results = await asyncio.gather(
        fetch_medical_news(topic),
        fetch_clinical_trials(topic),
        fetch_pharma_profile(topic),
        return_exceptions=True
    )

    news, trials, profile = results

    return {
        "news": news if not isinstance(news, Exception) else "News unavailable",
        "trials": trials if not isinstance(trials, Exception) else "Trials unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Profile unavailable"
    }


# ======================================
# STEP 5: LLM ANALYSIS + REPORT GENERATION
# ======================================
def analyse_text(text):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
            Analyze the following healthcare data and extract key clinical insights:
            Focus on treatments, effectiveness, and trends.

            {text}
            """
        }]
    )
    return response.choices[0].message.content


def generate_report(analysis, topic):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
            Create a professional medical research summary for: {topic}

            Include:
            - Key findings
            - Clinical insights
            - Industry relevance
            - Future outlook

            {analysis}
            """
        }]
    )
    return response.choices[0].message.content


# ======================================
# STEP 6: FULL MCP PIPELINE
# ======================================
async def full_pipeline(topic):

    # Step 1: Parallel Data Collection
    data = await parallel_research(topic)

    combined_text = f"""
    Medical News: {data['news']}
    Clinical Trials: {data['trials']}
    Pharma Profile: {data['profile']}
    """

    # Step 2: LLM Analysis
    analysis = analyse_text(combined_text)

    # Step 3: Report Generation
    report = generate_report(analysis, topic)

    return report


# ======================================
# STEP 7: RUN SYSTEM
# ======================================
topic = "Diabetes Treatment"

result = asyncio.run(full_pipeline(topic))

print("📊 HEALTHCARE RESEARCH REPORT:\n")
print(result)

📊 HEALTHCARE RESEARCH REPORT:

**Professional Medical Research Summary: Diabetes Treatment**

**Introduction:**
Diabetes is a chronic and debilitating condition that affects millions of people worldwide. Recent advancements in diabetes treatment have shown promising results, and this summary aims to provide an overview of the key findings, clinical insights, industry relevance, and future outlook in the field of diabetes treatment.

**Key Findings:**

1. **Emergence of New Therapies:** Recent clinical trials have introduced novel therapies that have demonstrated significant efficacy in managing diabetes.
2. **High Success Rate:** Clinical trials for diabetes treatment have achieved a remarkable success rate of approximately 93%, indicating that these new therapies are highly effective in controlling the condition.
3. **Increasing Emphasis on Innovation:** The growing trend towards innovative and effective solutions in diabetes treatment is likely to drive changes in clinical practice g

🖥️ SCENARIO: “AI IT Helpdesk Assistant in a Large Company”
🏢 Background Story

A large company (like Infosys or TCS) has thousands of employees.

👉 Employees face issues daily:

VPN not working
System hacked
Email access denied
Network outage

👉 Instead of manual IT support, the company builds an:

🤖 AI IT Helpdesk Assistant

In [3]:
# ======================================
# STEP 1: Install Dependencies
# ======================================
# pip install groq python-dotenv nest_asyncio


# ======================================
# STEP 2: Load API Key from .env
# ======================================
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Imports
# ======================================
import asyncio
import random
import logging

logging.basicConfig(level=logging.INFO)


# ======================================
# STEP 4: Mock MCP Tools
# ======================================
async def page_security_team(payload):
    return "🚨 Security team paged"

async def create_jira_ticket(payload):
    return f"🎫 Jira ticket created: {payload}"

async def check_network_status(payload):
    return random.choice(["Network degraded", "All systems normal"])

async def alert_noc_team(payload):
    return "📡 NOC team alerted"

async def get_ad_user(payload):
    return "user_123"

async def reset_permissions(payload):
    return "🔐 Permissions reset successfully"

async def query_postgres(payload):
    if random.random() < 0.7:
        raise Exception("Database timeout")
    return "📦 Data from Postgres"

async def query_sqlite_cache(payload):
    return "🗂 Data from SQLite cache (fallback)"


# ======================================
# STEP 5: LLM Classifier
# ======================================
def classify_issue(issue):
    prompt = f"""
    Classify the following IT issue into ONE category only:
    network, hardware, software, security, access

    Issue: {issue}

    Return only the category name.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip().lower()


# ======================================
# STEP 6: Smart Routing Logic
# ======================================
async def smart_it_triage(issue, severity):

    issue_type = classify_issue(issue)
    print(f"🧠 Classified as: {issue_type}")

    if issue_type == "security":
        await page_security_team({"issue": issue})
        return await create_jira_ticket({"project": "SEC", "priority": "Blocker"})

    elif issue_type == "network" and severity == "high":
        status = await check_network_status({})

        if "degraded" in status.lower():
            return await alert_noc_team({"issue": issue})
        else:
            return await create_jira_ticket({"project": "NET", "priority": "High"})

    elif issue_type == "access":
        user = await get_ad_user({"query": issue})
        return await reset_permissions({"user": user})

    else:
        return await create_jira_ticket({"project": "IT", "priority": "Medium"})


# ======================================
# STEP 7: Retry with Backoff
# ======================================
async def call_tool_with_retry(primary_tool, fallback_tool=None, max_retries=3):

    delay = 1

    for attempt in range(max_retries):
        try:
            return await primary_tool({})

        except Exception as e:
            logging.warning(f"Attempt {attempt+1} failed: {e}")

            if attempt == max_retries - 1:
                if fallback_tool:
                    logging.info("Switching to fallback tool")
                    return await fallback_tool({})
                raise

            await asyncio.sleep(delay)
            delay *= 2


# ======================================
# STEP 8: MAIN EXECUTION
# ======================================
async def run_demo():

    print("=== Smart IT Triage Demo ===\n")

    result = await smart_it_triage(
        issue="VPN not working for employee",
        severity="high"
    )

    print("🔍 Routing Result:", result)

    print("\n=== Retry Pattern Demo ===\n")

    data = await call_tool_with_retry(
        query_postgres,
        fallback_tool=query_sqlite_cache
    )

    print("📊 Data Result:", data)


# ======================================
# STEP 9: Run (PyCharm + Jupyter Safe)
# ======================================
import asyncio

try:
    loop = asyncio.get_running_loop()
    # If loop exists → Jupyter / Notebook
    import nest_asyncio
    nest_asyncio.apply()
    await run_demo()

except RuntimeError:
    # If no loop → normal Python script (PyCharm .py)
    asyncio.run(run_demo())

=== Smart IT Triage Demo ===



INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


🧠 Classified as: security
🔍 Routing Result: 🎫 Jira ticket created: {'project': 'SEC', 'priority': 'Blocker'}

=== Retry Pattern Demo ===



📊 Data Result: 📦 Data from Postgres


SCENARIO: “AI Healthcare Support Assistant in a Large Hospital”
🏢 Background Story
A large hospital (like Apollo or Fortis) has thousands of patients and staff members.
👉 Daily Challenges Faced:
- Patients waiting hours for appointment confirmations
- Confusion about lab test results and reports
- Doctors overwhelmed with scheduling and follow-up reminders
- Nurses struggling to track medicine administration times
- Emergency cases needing instant triage
👉 Instead of manual coordination, the hospital builds an:
🤖 AI Healthcare Support Assistant
🚑 Capabilities:
- 📅 Smart Scheduling: Automatically books and reschedules patient appointments based on doctor availability.
- 🧪 Lab Report Explainer: Summarizes test results in simple language for patients.
- 💊 Medication Tracker: Sends reminders to nurses and patients about dosage timings.
- 🚨 Emergency Triage: Instantly categorizes incoming cases (critical, urgent, routine) and alerts the right medical team.
- 📧 Follow-up Automation: Sends personalized recovery instructions and reminders after discharge.
🎯 Impact:
- Reduced patient waiting time
- Doctors spend more time on treatment, less on admin work
- Nurses avoid errors in medicine administration
- Faster response in emergencies
- Improved patient satisfaction and trust

In [4]:
# ======================================
# STEP 1: Install Dependencies
# ======================================
# pip install groq python-dotenv nest_asyncio


# ======================================
# STEP 2: Load API Key (.env)
# ======================================
import os
from dotenv import load_dotenv

load_dotenv()
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# ======================================
# STEP 3: Imports
# ======================================
import asyncio
import random
import logging

logging.basicConfig(level=logging.INFO)


# ======================================
# STEP 4: MOCK MCP TOOLS
# ======================================

# 📅 Scheduling
async def book_appointment(payload):
    return f"📅 Appointment booked for {payload.get('patient')} with Dr. Sharma"

# 🧪 Lab Report Explainer
async def explain_lab_report(payload):
    return "🧪 Report Summary: All parameters are within normal range."

# 💊 Medication Tracker
async def send_medication_reminder(payload):
    return "💊 Reminder sent for medication schedule"

# 🚨 Emergency Triage
async def alert_emergency_team(payload):
    return "🚨 Emergency team alerted immediately"

# 📧 Follow-up
async def send_followup(payload):
    return "📧 Follow-up instructions sent to patient"


# ======================================
# STEP 5: LLM CLASSIFIER (MCP BRAIN)
# ======================================
def classify_request(query):
    prompt = f"""
    Classify this healthcare request into ONE category:
    appointment, lab, medication, emergency, followup

    Query: {query}

    Return only the category.
    """

    res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return res.choices[0].message.content.strip().lower()


# ======================================
# STEP 6: SMART ROUTING LOGIC
# ======================================
async def healthcare_triage(query, severity="normal"):

    category = classify_request(query)
    print(f"🧠 Classified as: {category}")

    # 🚨 Emergency
    if "emergency" in category:
        return await alert_emergency_team({"query": query})

    # 📅 Appointment
    elif "appointment" in category:
        return await book_appointment({"patient": "Patient_X"})

    # 🧪 Lab
    elif "lab" in category:
        return await explain_lab_report({"report": query})

    # 💊 Medication
    elif "medication" in category:
        return await send_medication_reminder({})

    # 📧 Follow-up
    elif "followup" in category:
        return await send_followup({})

    else:
        return "🤖 Unable to classify request"


# ======================================
# STEP 7: RETRY MECHANISM (OPTIONAL TOOL)
# ======================================
async def call_with_retry(primary, fallback=None, retries=3):

    delay = 1

    for attempt in range(retries):
        try:
            return await primary({})

        except Exception as e:
            logging.warning(f"Retry {attempt+1} failed: {e}")

            if attempt == retries - 1 and fallback:
                return await fallback({})

            await asyncio.sleep(delay)
            delay *= 2


# ======================================
# STEP 8: DEMO RUNNER
# ======================================
async def run_demo():

    print("=== 🏥 Healthcare Assistant Demo ===\n")

    queries = [
        "Book appointment with cardiologist",
        "Explain my blood test report",
        "Remind me to take medicine",
        "Patient having heart attack",
        "Send discharge follow-up instructions"
    ]

    for q in queries:
        print(f"\n🗨️ Query: {q}")
        result = await healthcare_triage(q)
        print("👉 Response:", result)


# ======================================
# STEP 9: RUN (SAFE FOR BOTH ENV)
# ======================================
import asyncio

try:
    loop = asyncio.get_running_loop()
    import nest_asyncio
    nest_asyncio.apply()
    await run_demo()

except RuntimeError:
    asyncio.run(run_demo())

=== 🏥 Healthcare Assistant Demo ===


🗨️ Query: Book appointment with cardiologist


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


🧠 Classified as: appointment
👉 Response: 📅 Appointment booked for Patient_X with Dr. Sharma

🗨️ Query: Explain my blood test report
🧠 Classified as: followup
👉 Response: 📧 Follow-up instructions sent to patient

🗨️ Query: Remind me to take medicine


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


🧠 Classified as: medication
👉 Response: 💊 Reminder sent for medication schedule

🗨️ Query: Patient having heart attack


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


🧠 Classified as: emergency
👉 Response: 🚨 Emergency team alerted immediately

🗨️ Query: Send discharge follow-up instructions


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


🧠 Classified as: followup
👉 Response: 📧 Follow-up instructions sent to patient
